In [ ]:
import tushare as ts
import pandas as pd
import time
import os

# CORE CONFIGURATION (配置区)
# Note:  请在本地运行时切换你自己的TUSHARE的真实TOKEN!
MY_TOKEN = '换上你的真实TOKEN!' 
START_DATE = '20260101'  # 起始日
END_DATE = '20260611'    #同上
TARGET_DIR = 'G:/quant_data/daily_bar/market_data' # 别把你写网址的习惯带过来

# Clean up proxy environment to avoid potential connection dropping（开着代理运行不出错的保障代码）
os.environ['HTTP_PROXY'] = ''
os.environ['HTTPS_PROXY'] = ''

# Initialize Tushare Pro API
ts.set_token(MY_TOKEN)
pro = ts.pro_api()

if not os.path.exists(TARGET_DIR):
    os.makedirs(TARGET_DIR)

# Define our 10 target stocks from CSI300 (锁定10只沪深300核心股票)
target_stocks = ['000001', '000002', '600000', '600016', '600028', '600030', '600036', '600050', '600104', '600519']
ts_code_list = [f"{code}.SZ" if code.startswith('00') else f"{code}.SH" for code in target_stocks]

# STEP 1: FETCH TRADING CALENDAR (获取真实交易日历)
# Why? To skip weekends and holidays automatically, minimizing useless API requests.
# 逻辑：先拿日历，只对真正开盘的日子发起请求，完美避开不开市的日子
cal_df = pro.trade_cal(exchange='SSE', start_date=START_DATE, end_date=END_DATE, is_open=1)
trade_days = cal_df['cal_date'].tolist()
#加这个是怕你等半天不出反馈以为是代码有问题
print(f" Found {len(trade_days)} valid trading days. Starting pipeline...")
print("====================================================")

all_days_data = []

# STEP 2: CALENDAR-STEPPING LOOP (跨日横截面步进循环)
# Why daily loop instead of stock loop? Tushare pro.daily allows fetching ALL stocks in ONE single day.
# It drastically reduces the number of API requests compared to looping through hundreds of stocks.
# 逻辑：一天一天往下走，每一步直接拿全市场。单次敲门拿一堆，比一只只股票查高效百倍，还不会被坑人的获取限制卡
for date in trade_days:
    try:
        # Fetch cross-sectional snapshot for this specific date
        df_day = pro.daily(trade_date=date, fields='ts_code,trade_date,open,high,low,close,vol,amount')
        
        if df_day is not None and not df_day.empty:
            # Filter targeted 10 stocks locally to save computation
            df_filter = df_day[df_day['ts_code'].isin(ts_code_list)]
            all_days_data.append(df_filter)
            print(f"   ✅ Date {date}: Aligned {len(df_filter)} target stocks successfully.")
            
        # Defensive pacing (0.1s delay) to respect rate limit caps
        time.sleep(0.1)
    except Exception as e:
        print(f"Error on Date {date}: {e}")

print("====================================================")
print("Transforming long logs into 2D cross-sectional tensors")

#STEP 3: PANDAS PIVOT MATRIX INTERSECTION (矩阵二维化透视)
if all_days_data:
    # Combine all standard data frames into one massive master log
    df_total = pd.concat(all_days_data, ignore_index=True)
    df_total['pure_code'] = df_total['ts_code'].apply(lambda x: x.split('.')[0])
    df_total['trade_date'] = pd.to_datetime(df_total['trade_date'], format='%Y%m%d')
    
    # MAGIC: Reshape table so that Index=Date, Columns=Stock Tickers, Values=Financial Metrics
    # 核心：用 pivot_table 一键将“大长表”揉成标准量化 2D 大矩阵（横截面对齐）
    close_matrix    = df_total.pivot_table(index='trade_date', columns='pure_code', values='close').sort_index()
    open_matrix     = df_total.pivot_table(index='trade_date', columns='pure_code', values='open').sort_index()
    high_matrix     = df_total.pivot_table(index='trade_date', columns='pure_code', values='high').sort_index()
    low_matrix      = df_total.pivot_table(index='trade_date', columns='pure_code', values='low').sort_index()
    vol_matrix      = df_total.pivot_table(index='trade_date', columns='pure_code', values='vol').sort_index()
    turnover_matrix = df_total.pivot_table(index='trade_date', columns='pure_code', values='amount').sort_index() 
    
    #STEP 4: PREVENT LOOK-AHEAD BIAS (彻底隔离未来函数)
    # `.shift(-1)` shifts the return matrix UPWARDS by 1 row.
    # Row '2026-06-01' will contain the actual percentage return of '2026-06-02'.
    # This allows us to use features at market close of Day T to predict the future target of Day T+1.
    # 逻辑：通过 .shift(-1) 将收益率整体往上提一行。用今天收盘看到的特征，去预测明天的涨跌。这个挺核心的别把关键的y_target搞错了
    daily_return = close_matrix.pct_change()
    y_target_matrix = daily_return.shift(-1)
    
    # Uniformly name the row index
    for df in [close_matrix, open_matrix, high_matrix, low_matrix, vol_matrix, turnover_matrix, y_target_matrix]:
        df.index.name = 'Date'
        
    #STEP 5: SOLID LOCAL STORAGE (落盘物理锁定)
    close_matrix.to_csv(os.path.join(TARGET_DIR, 'close_matrix.csv'))
    open_matrix.to_csv(os.path.join(TARGET_DIR, 'open_matrix.csv'))
    high_matrix.to_csv(os.path.join(TARGET_DIR, 'high_matrix.csv'))
    low_matrix.to_csv(os.path.join(TARGET_DIR, 'low_matrix.csv'))
    vol_matrix.to_csv(os.path.join(TARGET_DIR, 'vol_matrix.csv'))
    turnover_matrix.to_csv(os.path.join(TARGET_DIR, 'turnover_matrix.csv'))
    y_target_matrix.to_csv(os.path.join(TARGET_DIR, 'y_target_matrix.csv'))
    
    print(f"\nPipeline phase 1 complete! Captured {len(close_matrix)} continuous trading rows!")
    print(f"Matrices successfully synchronized to local directory: {TARGET_DIR}")